### Links and Resources
| Method / Class | Description |
|----------------|-------------|
| [DeltaTable](https://docs.delta.io/latest/api/python/spark/index.html) | Entry point for Delta Lake table operations |
| [DeltaTable.forName()](https://docs.delta.io/latest/api/python/spark/index.html) | Loads a Delta table by its Unity Catalog name (`catalog.schema.table`) |
| [DeltaTable.forPath()](https://docs.delta.io/latest/api/python/spark/index.html) | Loads a Delta table by its storage path (DBFS or Volume path) |
| [DeltaTable.toDF()](https://docs.delta.io/latest/api/python/spark/index.html) | Returns the Delta table as a Spark DataFrame |
| [DeltaTable.history()](https://docs.delta.io/latest/api/python/spark/index.html) | Returns a DataFrame with the transaction log history of the table |
| [DeltaTable.detail()](https://docs.delta.io/latest/api/python/spark/index.html) | Returns metadata about the Delta table (size, partitioning, format, etc.) |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()


def display(df, n=20):
    df.show(n, truncate=False)

# Delta Table Introduction

**Delta Lake** is an open-source storage layer that brings ACID transactions, schema enforcement, and time travel to Apache Spark workloads. Every Delta table is backed by a `_delta_log/` transaction log that records every operation applied to the table.

The `DeltaTable` class is the main Python entry point for interacting with Delta tables programmatically. There are two ways to load an existing Delta table:

| Method | When to use |
|--------|-------------|
| `DeltaTable.forName(spark, "catalog.schema.table")` | Table is registered in Unity Catalog (or the Hive metastore) |
| `DeltaTable.forPath(spark, "/path/to/delta/table")` | You know the physical storage path of the Delta table |

### Importing DeltaTable

`DeltaTable` lives in the `delta.tables` package, which is pre-installed on every Databricks cluster with Delta Lake support.

In [ ]:
from delta.tables import DeltaTable

### `DeltaTable.forName()` — Load by Catalog Name

`forName` resolves the table through Unity Catalog using the three-part identifier `catalog.schema.table`.  
This is the preferred approach when tables are managed by Unity Catalog because it is location-independent — you do not need to know where the data is physically stored.

In [ ]:
# Load the country_regions managed table from Unity Catalog
# Catalog: population_metrics | Schema: default | Table: country_regions
dt_by_name = DeltaTable.forName(spark, "population_metrics.default.country_regions")

print(type(dt_by_name))  # delta.tables.DeltaTable

# Convert to a regular Spark DataFrame for querying
df_by_name = dt_by_name.toDF()
display(df_by_name)

### Inspect Schema and Metadata via `forName`

`detail()` returns a single-row DataFrame with table-level metadata: location, format, creation time, number of files, and more.

In [ ]:
# Print the schema of the underlying DataFrame
print("Schema:")
df_by_name.printSchema()

# detail() gives metadata about the Delta table itself
print("\nTable detail:")
display(dt_by_name.detail())

### Transaction History via `forName`

`history()` reads the `_delta_log/` transaction log and returns one row per commit. Each row shows the operation type (`WRITE`, `CREATE TABLE`, `MERGE`, etc.), timestamp, user, and Spark version.

In [ ]:
# Show the last 10 operations on the country_regions table
display(dt_by_name.history(10).select("version", "timestamp", "operation", "operationParameters"))

### `DeltaTable.forPath()` — Load by Storage Path

`forPath` bypasses Unity Catalog and reads the Delta table directly from its storage location.  
This is useful when:
- The table is not registered in a catalog (external or legacy tables).
- You are working with a path in a Unity Catalog Volume or DBFS.
- You need to access a specific physical location regardless of catalog registration.

The data at the countries_population CSV path will first be written as a **Delta table** to a location in the Volume, and then accessed with `forPath`.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

csv_path = (
    "/Volumes/population_metrics/landing/datasets/countries_dataset/"
    "countries_dataset/csv_data/countries_population/countries_population.csv"
)

schema = StructType([
    StructField("COUNTRY_ID",    IntegerType(), True),
    StructField("NAME",          StringType(),  True),
    StructField("NATIONALITY",   StringType(),  True),
    StructField("CODE",          StringType(),  True),
    StructField("ISO_ALPHA3",    StringType(),  True),
    StructField("CAPITAL",       StringType(),  True),
    StructField("POPULATION",    IntegerType(), True),
    StructField("AREA_KM2",      DoubleType(),  True),
    StructField("REGION_ID",     IntegerType(), True),
    StructField("SUB_REGION_ID", IntegerType(), True),
])

# Read the source CSV
df_csv = spark.read.format("csv").schema(schema).option("header", True).load(csv_path)

print(f"Rows read from CSV: {df_csv.count()}")
display(df_csv)

### Write the CSV Data as a Delta Table to a Volume Path

Delta tables written to a Volume path are not automatically registered in Unity Catalog — they exist only at the physical path.  
This is exactly the scenario where `forPath` is the right tool.

In [ ]:
# Target path for the Delta table inside the Volume
delta_path = "/Volumes/population_metrics/landing/datasets/countries_dataset/countries_dataset/delta/countries_population_delta"

# Write as Delta format (overwrite if it already exists)
df_csv.write.format("delta").mode("overwrite").save(delta_path)

print(f"Delta table written to: {delta_path}")

### Load the Delta Table via `forPath`

Now that a Delta table exists at the Volume path, `DeltaTable.forPath` can load it directly without any catalog registration.

In [ ]:
# Load the Delta table by its physical storage path
dt_by_path = DeltaTable.forPath(spark, delta_path)

print(type(dt_by_path))  # delta.tables.DeltaTable

# Convert to DataFrame for querying
df_by_path = dt_by_path.toDF()
display(df_by_path)

### Inspect Schema and Metadata via `forPath`

In [ ]:
# Schema of the Delta table loaded by path
print("Schema:")
df_by_path.printSchema()

# Metadata: size, location, format details
print("\nTable detail:")
display(dt_by_path.detail())

### Transaction History via `forPath`

The transaction log works identically whether the table was loaded by name or by path.

In [ ]:
display(dt_by_path.history().select("version", "timestamp", "operation", "operationParameters"))

### Summary: `forName` vs `forPath`

| | `forName` | `forPath` |
|-|-----------|-----------|
| **Input** | `"catalog.schema.table"` | Physical storage path |
| **Requires catalog registration** | Yes | No |
| **Location-independent** | Yes | No |
| **Works with external/unregistered tables** | No | Yes |
| **Preferred for Unity Catalog managed tables** | Yes | No |

Both methods return a `DeltaTable` object with the same API: `toDF()`, `detail()`, `history()`, `update()`, `delete()`, `merge()`, and more.